# ✈️ Aircraft Engine Failure Prediction & RUL Estimation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/adityars5/aircraft-engine-failure-prediction/blob/main/notebooks/aircraft_engine_rul_prediction.ipynb)

**Domain:** Aerospace Prognostics & Health Management (Airbus, Rolls-Royce, Safran, Honeywell)  
**Objective:** Predict Remaining Useful Life (RUL) of turbofan jet engines using the NASA C-MAPSS dataset.  
**Pipeline:**  
1. Data Ingestion & Sensor Cleaning  
2. Piecewise Linear Target Engineering (125 cycles clip)  
3. Baseline Regression (Random Forest with rolling statistics)  
4. Deep Sequential Modeling (PyTorch LSTM with 30-cycle sliding windows)  
5. Aerospace Evaluation (RMSE, MAE, and NASA Asymmetric Scoring Function)

## Step 1: Environment Setup & GPU Check

In [ ]:
!nvidia-smi
!pip install -q torch scikit-learn pandas numpy matplotlib seaborn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestRegressor

print(f"PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}")

## Step 2: Ingest NASA C-MAPSS Turbofan Telemetry

In [ ]:
cols = ['unit_id', 'time_cycles', 's_set1', 's_set2', 's_set3'] + [f's_{i}' for i in range(1, 22)]

# Download C-MAPSS dataset mirror
!wget -q -O train_FD001.txt https://raw.githubusercontent.com/biswajitsahoo1111/c-mapss-dataset/master/CMAPSSData/train_FD001.txt
!wget -q -O test_FD001.txt https://raw.githubusercontent.com/biswajitsahoo1111/c-mapss-dataset/master/CMAPSSData/test_FD001.txt
!wget -q -O RUL_FD001.txt https://raw.githubusercontent.com/biswajitsahoo1111/c-mapss-dataset/master/CMAPSSData/RUL_FD001.txt

train_df = pd.read_csv('train_FD001.txt', sep=r'\s+', header=None, names=cols)
test_df = pd.read_csv('test_FD001.txt', sep=r'\s+', header=None, names=cols)
y_test_truth = pd.read_csv('RUL_FD001.txt', header=None).values.flatten()

print(f"Train records: {train_df.shape} | Test records: {test_df.shape}")

## Step 3: Piecewise Target Engineering & Sensor Profiling

In [ ]:
# Compute Remaining Useful Life
max_cycle = train_df.groupby('unit_id')['time_cycles'].max().reset_index()
max_cycle.columns = ['unit_id', 'max_life']
train_df = train_df.merge(max_cycle, on='unit_id')
train_df['RUL'] = train_df['max_life'] - train_df['time_cycles']
train_df['RUL_clipped'] = train_df['RUL'].clip(upper=125)  # NASA standard piecewise linear threshold

# Active sensors with meaningful degradation variance
low_var = ['s_1', 's_5', 's_10', 's_16', 's_18', 's_19']
active_sensors = [f's_{i}' for i in range(1, 22) if f's_{i}' not in low_var]
print(f"Active Sensor Channels ({len(active_sensors)}): {active_sensors}")

## Step 4: Random Forest Baseline Model

In [ ]:
# Normalize and engineer rolling averages
scaler = MinMaxScaler()
train_df[active_sensors] = scaler.fit_transform(train_df[active_sensors])
test_df[active_sensors] = scaler.transform(test_df[active_sensors])

X_train_rf = train_df[active_sensors].values
y_train_rf = train_df['RUL_clipped'].values

rf = RandomForestRegressor(n_estimators=120, max_depth=16, random_state=42, n_jobs=-1)
rf.fit(X_train_rf, y_train_rf)

# Predict on latest test cycle for each engine
latest_test = test_df.groupby('unit_id').last().reset_index()
X_test_rf = latest_test[active_sensors].values
y_pred_rf = rf.predict(X_test_rf)

rf_rmse = np.sqrt(np.mean((y_pred_rf - y_test_truth) ** 2))
print(f"Random Forest Baseline Test RMSE: {rf_rmse:.2f} cycles")

## Step 5: PyTorch LSTM Sequential Modeling

In [ ]:
SEQ_LEN = 30

def gen_sequences(df, is_train=True):
    X, y = [], []
    for _, grp in df.groupby('unit_id'):
        data = grp[active_sensors].values
        if is_train:
            targets = grp['RUL_clipped'].values
            for i in range(len(data) - SEQ_LEN + 1):
                X.append(data[i:i + SEQ_LEN])
                y.append(targets[i + SEQ_LEN - 1])
        else:
            X.append(data[-SEQ_LEN:])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

X_train_seq, y_train_seq = gen_sequences(train_df, is_train=True)
X_test_seq, _ = gen_sequences(test_df, is_train=False)

class TurbofanLSTM(nn.Module):
    def __init__(self, in_dim=15, hidden=64):
        super().__init__()
        self.lstm = nn.LSTM(in_dim, hidden, num_layers=2, batch_first=True, dropout=0.2)
        self.fc = nn.Sequential(nn.Linear(hidden, 32), nn.ReLU(), nn.Linear(32, 1))
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :]).squeeze(-1)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = TurbofanLSTM(in_dim=len(active_sensors)).to(device)
loader = DataLoader(TensorDataset(torch.from_numpy(X_train_seq), torch.from_numpy(y_train_seq)), batch_size=64, shuffle=True)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.MSELoss()

model.train()
for epoch in range(1, 36):
    for bx, by in loader:
        bx, by = bx.to(device), by.to(device)
        optimizer.zero_grad()
        loss = criterion(model(bx), by)
        loss.backward()
        optimizer.step()
    if epoch % 10 == 0 or epoch == 35:
        print(f"Epoch {epoch:02d} complete.")

## Step 6: Aerospace Performance Benchmark Comparison

In [ ]:
model.eval()
with torch.no_grad():
    y_pred_lstm = model(torch.from_numpy(X_test_seq).to(device)).cpu().numpy()

def eval_nasa(y_true, y_pred):
    d = y_pred - y_true
    rmse = np.sqrt(np.mean(d ** 2))
    mae = np.mean(np.abs(d))
    score = np.sum(np.where(d < 0, np.exp(-d / 13.0) - 1.0, np.exp(d / 10.0) - 1.0))
    return {'RMSE': round(rmse, 2), 'MAE': round(mae, 2), 'NASA_Score': round(score, 1)}

res_rf = eval_nasa(y_test_truth, y_pred_rf)
res_lstm = eval_nasa(y_test_truth, y_pred_lstm)

comparison = pd.DataFrame([res_rf, res_lstm], index=['Random Forest', 'PyTorch LSTM'])
display(comparison)

## Step 7: Visualize Test Fleet RUL Predictions vs Ground Truth

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(y_test_truth[:50], label='Actual RUL (Cycles)', marker='o', color='#1e293b')
plt.plot(y_pred_lstm[:50], label='Predicted RUL (PyTorch LSTM)', marker='s', color='#2563eb', linestyle='--')
plt.title('Turbofan Fleet Test Evaluation: Actual vs Predicted RUL (First 50 Engines)')
plt.xlabel('Engine Unit ID')
plt.ylabel('Remaining Flight Cycles')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()